# Домашнее задание 1. Вековые возмущения от $J_2$

Дополните этот ноутбук полным решением обоих заданий: добавьте код расчётов и
построения графиков, аналитические выводы и ответы на вопросы в текстовых ячейках.
Перед отправкой выполните все ячейки по порядку и сохраните файл с результатами
вычислений и графиками. Сдаётся один файл `hw_1.ipynb`.

## Пространственная визуализация к заданию 2

Визуализация к заданию 2. Одновременно представлены три орбиты с наклонениями
$50^\circ$, $90^\circ$ и критическим наклонением $i<90^\circ$.
Для всех орбит заданы одинаковые
$a=10\,000$ км, $e=0.3$, $\Omega_0=20^\circ$, $\omega_0=40^\circ$.

Используется модель вековой эволюции **средних элементов в первом порядке по $J_2$**.
Большая полуось, эксцентриситет и наклонение постоянны. Показанный на каждой эпохе
эллипс представляет геометрию орбиты при текущих элементах; **это не траектория КА
за рассматриваемый интервал времени**. Положение КА на орбите здесь не вычисляется.

Выполните ячейки последовательно. Готовый код пространственной визуализации
изменять не требуется. Рисунки для эпох 0, 90 и 180 суток выводятся автоматически
и сохраняются в результатах ячейки при сохранении ноутбука.
При наличии `ipywidgets` дополнительно доступен ползунок для выбора эпохи
от 0 до 180 суток. Если среда не отображает виджеты, используйте статические рисунки.

Для запуска в выбранном окружении Python должны быть установлены `numpy`,
`matplotlib` и ядро Jupyter `ipykernel`. Пакет `ipywidgets` необязателен.

Сопоставьте изменение положения орбитальной плоскости с изменением ориентации
эллипса относительно линии узлов. Для наблюдения используйте одинаковые эпохи
на всех трёх изображениях. Аналитические выводы и ответы оформите в отдельных текстовых ячейках этого ноутбука.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Все длины в расчёте — километры, время — секунды, углы — радианы.
MU = 398600.4418
EARTH_RADIUS = 6378.1363
J2 = 1.08262668e-3
SECONDS_PER_DAY = 86400.0

SEMIMAJOR_AXIS = 10000.0
ECCENTRICITY = 0.3
RAAN_0 = np.deg2rad(20.0)
ARGUMENT_OF_PERICENTER_0 = np.deg2rad(40.0)
INCLINATIONS = np.array([
    np.deg2rad(50.0),
    np.deg2rad(90.0),
    np.deg2rad(63.434948823),
])

USE_WIDGETS = True
SNAPSHOT_DAYS = (0.0, 90.0, 180.0)
VIEW_ELEVATION = 20.0
VIEW_AZIMUTH = -40.0

COLORS = {
    "paper": "#FBF9F3",
    "ink": "#302B28",
    "muted": "#6D6560",
    "accent": "#187AA5",
    "dark_blue": "#095C83",
    "equator": "#B8BABD",
    "nodes": "#56646E",
}
font_names = {item.name for item in font_manager.fontManager.ttflist}
COURSE_FONT = "PT Serif" if "PT Serif" in font_names else "DejaVu Serif"
plt.rcParams.update({
    "font.family": COURSE_FONT,
    "font.size": 11,
    "text.color": COLORS["ink"],
    "figure.facecolor": COLORS["paper"],
    "axes.facecolor": COLORS["paper"],
    "savefig.facecolor": COLORS["paper"],
    "mathtext.fontset": "dejavuserif",
})

## Геометрия и изменение элементов

Начало инерциальной системы координат находится в центре Земли. Плоскость $XY$
совпадает с экваториальной плоскостью; ось $Z$ направлена к северному полюсу.
Ось $X$ задаёт начало отсчёта $\Omega$.

Сплошная голубая кривая — орбита на выбранную эпоху. **ВУ** — восходящий узел,
**П** — перицентр, **А** — апоцентр. Штриховая линия узлов проходит через оба
пересечения орбиты с экватором; её положительное направление задаётся восходящим
узлом. Точечная линия соединяет апоцентр и перицентр и представляет линию апсид.
Жирный голубой отрезок направлен из центра Земли к перицентру.

Следующие ячейки содержат вспомогательные расчёты и построение сцены.

In [ ]:
def secular_orientation(inclination, days):
    """Ориентация орбиты в принятой модели; углы возвращаются в радианах."""
    a = SEMIMAJOR_AXIS
    e = ECCENTRICITY
    p = a * (1.0 - e**2)
    mean_motion = np.sqrt(MU / a**3)
    scale = J2 * mean_motion * (EARTH_RADIUS / p)**2
    cos_i = np.cos(inclination)
    raan_rate = -1.5 * scale * cos_i
    pericenter_rate = 0.75 * scale * (5.0 * cos_i**2 - 1.0)
    seconds = float(days) * SECONDS_PER_DAY
    return (
        RAAN_0 + raan_rate * seconds,
        ARGUMENT_OF_PERICENTER_0 + pericenter_rate * seconds,
    )


def orbital_geometry(inclination, days, samples=481):
    """Векторы и точки в инерциальной СК; первая строка кривой — перицентр."""
    raan, argument = secular_orientation(inclination, days)
    node_direction = np.array([np.cos(raan), np.sin(raan), 0.0])
    transverse_direction = np.array([
        -np.sin(raan) * np.cos(inclination),
        np.cos(raan) * np.cos(inclination),
        np.sin(inclination),
    ])
    pericenter_direction = (
        np.cos(argument) * node_direction
        + np.sin(argument) * transverse_direction
    )
    quadrature_direction = (
        -np.sin(argument) * node_direction
        + np.cos(argument) * transverse_direction
    )
    a, e = SEMIMAJOR_AXIS, ECCENTRICITY
    p = a * (1.0 - e**2)
    anomaly = np.linspace(0.0, 2.0 * np.pi, samples)
    radius = p / (1.0 + e * np.cos(anomaly))
    orbit = radius[:, None] * (
        np.cos(anomaly)[:, None] * pericenter_direction
        + np.sin(anomaly)[:, None] * quadrature_direction
    )
    ascending_radius = p / (1.0 + e * np.cos(argument))
    descending_radius = p / (1.0 - e * np.cos(argument))
    radial_derivative = -p * e * np.sin(argument) / (1.0 + e * np.cos(argument))**2
    ascending_tangent = radial_derivative * node_direction + ascending_radius * transverse_direction
    ascending_tangent /= np.linalg.norm(ascending_tangent)
    return {
        "orbit": orbit,
        "node_direction": node_direction,
        "transverse_direction": transverse_direction,
        "pericenter_direction": pericenter_direction,
        "quadrature_direction": quadrature_direction,
        "normal": np.cross(node_direction, transverse_direction),
        "pericenter": a * (1.0 - e) * pericenter_direction,
        "apocenter": -a * (1.0 + e) * pericenter_direction,
        "ascending_node": ascending_radius * node_direction,
        "descending_node": -descending_radius * node_direction,
        "ascending_tangent": ascending_tangent,
        "raan": raan,
        "argument": argument,
    }

In [ ]:
def plane_patch(ax, basis_1, basis_2, radius, color, alpha):
    angles = np.linspace(0.0, 2.0 * np.pi, 97)
    edge = radius * (
        np.cos(angles)[:, None] * basis_1
        + np.sin(angles)[:, None] * basis_2
    )
    ax.add_collection3d(Poly3DCollection(
        [edge], facecolors=color, edgecolors=color,
        linewidths=0.55, alpha=alpha,
    ))


def draw_scene(ax, inclination, days):
    geometry = orbital_geometry(inclination, days)
    # Для рисунка координаты выражаются в тысячах километров.
    scale = 1000.0
    extent = 1.15 * SEMIMAJOR_AXIS * (1.0 + ECCENTRICITY) / scale
    plane_radius = 0.92 * extent
    orbit = geometry["orbit"] / scale
    origin = np.zeros(3)

    plane_patch(ax, np.eye(3)[0], np.eye(3)[1], plane_radius,
                COLORS["equator"], 0.19)
    plane_patch(ax, geometry["node_direction"], geometry["transverse_direction"],
                plane_radius, COLORS["accent"], 0.075)

    # Три явно обозначенные инерциальные оси, одинаковые во всех сценах.
    for direction, label in zip(np.eye(3), ("X", "Y", "Z")):
        segment = np.vstack((-0.97 * extent * direction, extent * direction))
        ax.plot(*segment.T, color=COLORS["muted"], linewidth=0.75, alpha=0.72)
        ax.quiver(*(0.80 * extent * direction), *(0.20 * extent * direction),
                  color=COLORS["muted"], linewidth=0.9, arrow_length_ratio=0.4)
        ax.text(*(1.055 * extent * direction), label,
                color=COLORS["ink"], fontsize=11, ha="center", va="center")

    node_line = extent * np.vstack((
        -geometry["node_direction"], geometry["node_direction"]
    ))
    ax.plot(*node_line.T, color=COLORS["nodes"], linewidth=1.6,
            linestyle=(0, (5, 3)))
    apsides = np.vstack((geometry["apocenter"], geometry["pericenter"])) / scale
    ax.plot(*apsides.T, color=COLORS["accent"], linewidth=1.4,
            linestyle=(0, (1, 2)))
    ax.plot(*orbit.T, color=COLORS["accent"], linewidth=2.1)

    pericenter = geometry["pericenter"] / scale
    ax.quiver(*origin, *pericenter, color=COLORS["dark_blue"],
              linewidth=2.0, arrow_length_ratio=0.08)
    ax.scatter(*origin, color=COLORS["ink"], s=18, depthshade=False)

    point_specs = (
        ("ascending_node", "ВУ", "o", COLORS["nodes"], 30, 1.0),
        ("pericenter", "П", "D", COLORS["dark_blue"], 24, -1.0),
        ("apocenter", "А", "o", COLORS["accent"], 19, 1.0),
    )
    for key, label, marker, color, size, vertical_offset in point_specs:
        point = geometry[key] / scale
        ax.scatter(*point, marker=marker, color=color, s=size, depthshade=False)
        if key == "pericenter":
            label_point = point + 0.09 * extent * geometry["pericenter_direction"]
        else:
            label_point = point + np.array([0.0, 0.0, 0.075 * extent * vertical_offset])
        ax.text(*label_point, label, fontsize=9, color=color, ha="center",
                va="bottom" if vertical_offset > 0 else "top",
                bbox={"facecolor": COLORS["paper"], "edgecolor": "none",
                      "alpha": 0.85, "pad": 0.8})
    descending = geometry["descending_node"] / scale
    ax.scatter(*descending, marker="o", facecolors=COLORS["paper"],
               edgecolors=COLORS["nodes"], s=22, linewidths=1, depthshade=False)

    # Короткая касательная стрелка у ВУ показывает переход к северу от экватора.
    tangent_at_node = geometry["ascending_tangent"]
    node = geometry["ascending_node"] / scale
    ax.quiver(*(node - 0.07 * extent * tangent_at_node),
              *(0.14 * extent * tangent_at_node),
              color=COLORS["nodes"], linewidth=1.0, arrow_length_ratio=0.22)

    ax.set_xlim(-extent, extent)
    ax.set_ylim(-extent, extent)
    ax.set_zlim(-extent, extent)
    ax.set_box_aspect((1, 1, 1), zoom=1.45)
    ax.view_init(elev=VIEW_ELEVATION, azim=VIEW_AZIMUTH)
    ax.set_proj_type("ortho")
    ax.set_axis_off()
    angle_degrees = np.rad2deg(inclination)
    angle_label = f"{angle_degrees:.5f}".rstrip("0").rstrip(".")
    ax.set_title(rf"$i={angle_label}^\circ$", fontsize=14, pad=4)
    raan_degrees = np.rad2deg(geometry["raan"]) % 360.0
    argument_degrees = np.rad2deg(geometry["argument"]) % 360.0
    ax.text2D(0.5, 0.02,
              rf"$\Omega={raan_degrees:.1f}^\circ,\quad\omega={argument_degrees:.1f}^\circ$",
              transform=ax.transAxes, ha="center", fontsize=11)
    return geometry


def render_orbits(days):
    """Три орбиты при одной эпохе; возвращает Figure, не сохраняет файлов."""
    fig = plt.figure(figsize=(15.2, 6.3), dpi=115)
    fig.suptitle(f"Геометрия орбит на эпоху t = {float(days):g} суток",
                 fontsize=16, y=0.96)
    fig.text(0.5, 0.902,
             "Инерциальная система координат; одинаковые масштаб и направление наблюдения",
             ha="center", color=COLORS["muted"], fontsize=10)
    for position, inclination in enumerate(INCLINATIONS, start=1):
        axis = fig.add_subplot(1, 3, position, projection="3d")
        draw_scene(axis, inclination, days)
    legend = [
        Line2D([0], [0], color=COLORS["accent"], lw=2.1, label="Орбита"),
        Line2D([0], [0], color=COLORS["nodes"], lw=1.6, ls="--", label="Линия узлов"),
        Line2D([0], [0], color=COLORS["accent"], lw=1.4, ls=":", label="Линия апсид"),
        Line2D([0], [0], color=COLORS["dark_blue"], lw=2, label="Радиус перицентра"),
        Patch(facecolor=COLORS["equator"], alpha=0.45, label="Экваториальная плоскость"),
        Patch(facecolor=COLORS["accent"], alpha=0.18, label="Плоскость орбиты"),
    ]
    fig.legend(handles=legend, loc="lower center", bbox_to_anchor=(0.5, 0.035),
               ncol=3, frameon=False, fontsize=10, columnspacing=2.2)
    fig.text(0.5, 0.015, "ВУ — восходящий узел; П — перицентр; А — апоцентр",
             ha="center", fontsize=9.5, color=COLORS["muted"])
    fig.subplots_adjust(left=0.015, right=0.985, top=0.85, bottom=0.19, wspace=0.015)
    return fig

## Выбор эпохи

Ползунок изменяет только время прогноза; параметры орбит заданы в первой кодовой ячейке.
Углы под каждым изображением приведены в диапазоне $[0^\circ,360^\circ)$.
Переход через границу этого диапазона не означает скачка ориентации орбиты.

Для получения отдельного изображения можно вызвать, например,
`render_orbits(45.0)` и `plt.show()`. Направление наблюдения задаётся параметрами
`VIEW_ELEVATION` и `VIEW_AZIMUTH` в первой кодовой ячейке (градусы).
Ноутбук не создаёт дополнительных файлов.

In [ ]:
# Обязательные рисунки сохраняются как обычные результаты ячейки,
# независимо от наличия и поддержки виджетов.
for days in SNAPSHOT_DAYS:
    figure = render_orbits(days)
    plt.show()
    plt.close(figure)

# Ползунок — дополнительный способ выбора эпохи.
widgets_available = False
if USE_WIDGETS:
    try:
        from IPython import get_ipython
        from IPython.display import display, clear_output
        import ipywidgets as widgets
        shell = get_ipython()
        widgets_available = shell is not None and getattr(shell, "kernel", None) is not None
    except ImportError:
        widgets_available = False

if widgets_available:
    time_slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=180.0, step=1.0,
        description="t, сутки:", continuous_update=False,
        readout_format=".0f", layout=widgets.Layout(width="85%"),
        style={"description_width": "80px"},
    )
    scene_output = widgets.Output()

    def update_scene(change=None):
        with scene_output:
            clear_output(wait=True)
            figure = render_orbits(time_slider.value)
            display(figure)
            plt.close(figure)

    time_slider.observe(update_scene, names="value")
    display(widgets.VBox([time_slider, scene_output]))
    update_scene()
